In [5]:
# sanity_check_same_qid_different_queries.py  ← 直接运行这一个！

import pandas as pd
import json

print("="*100)
print("终极检查：同一个 Question_ID → 多个不同的 user_query + 完全相同的 true_a_i")
print("="*100)

def load_jsonl_safely(path):
    """完美兼容所有 pandas 新版本，彻底消灭 FutureWarning 和 ValueError"""
    if not pd.io.common.os.path.exists(path):
        raise FileNotFoundError(f"文件不存在: {path}")
    
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"第 {i} 行 JSON 解析失败: {e}")
                print(f"   → {line[:100]}...")
                raise
    return pd.DataFrame(data)

# 主检查逻辑
for filename in ["train_final.jsonl", "test_final.jsonl"]:
    print(f"\n正在加载并检查 → {filename}")
    try:
        df = load_jsonl_safely(filename)
    except Exception as e:
        print(f"加载失败: {e}")
        continue

    # 必须列检查
    required = ['Question_ID', 'user_query', 'true_a_i']
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"缺少关键列: {missing}")
        continue

    # 统一类型
    df['Question_ID'] = df['Question_ID'].astype(str)
    df['user_query']  = df['user_query'].astype(str).str.strip()
    df['true_a_i']    = df['true_a_i'].astype(str).str.strip()

    print(f"  样本总数       : {len(df):,}")
    print(f"  唯一问题数     : {df['Question_ID'].nunique():,}")

    # 逐个检查每个 Question_ID
    bad_cases = []
    good_cases = 0

    for qid, group in df.groupby('Question_ID'):
        queries = group['user_query'].unique()
        answers = group['true_a_i'].unique()
        n = len(group)

        if n == 1:
            bad_cases.append((qid, "仅1条样本", queries[0], ""))
        elif len(queries) < n:
            bad_cases.append((qid, "有重复query", f"{len(queries)}种/{n}条", queries.tolist()[:3]))
        elif len(answers) > 1:
            bad_cases.append((qid, "答案不一致", f"{len(answers)}种答案", answers.tolist()[:2]))
        else:
            good_cases += 1

    print(f"  完美题目数     : {good_cases:,}  ({good_cases/df['Question_ID'].nunique():.1%})")
    print(f"  有问题题目数   : {len(bad_cases):,}")

    if bad_cases:
        print("\n  问题样例（前10个）：")
        for i, (qid, typ, info, detail) in enumerate(bad_cases[:10], 1):
            print(f"    {i:2}. Question_ID={qid}  →  {typ}")
            if isinstance(detail, list):
                for q in detail:
                    print(f"        • {q}")
            else:
                print(f"        {info}")
    else:
        print("  完美！所有 Question_ID 都满足：多条不同 user_query + 完全一致的 true_a_i")

print("\n" + "="*100)
print("检查完成！现在你可以放心了")
print("如果 99%+ 题目显示 '完美' → 数据结构完全正确，可以直接训练！")

终极检查：同一个 Question_ID → 多个不同的 user_query + 完全相同的 true_a_i

正在加载并检查 → train_final.jsonl
  样本总数       : 29,163
  唯一问题数     : 8,803
  完美题目数     : 8,496  (96.5%)
  有问题题目数   : 307

  问题样例（前10个）：
     1. Question_ID=5593  →  答案不一致
        • The current head of NASA is James Frederick Bridenstine [
        • ].
     2. Question_ID=8461  →  有重复query
        • To what extent has the resident count been estimated for Massachusetts?
        • Do statistics exist that detail the population count within Massachusetts?
        • How many individuals reside within Massachusetts borders?
     3. Question_ID=8462  →  有重复query
        • Did some unknown epoch see the advent of symbolic representations of sound through writing?
        • In ancient times or more modern era, when did musical expression take shape?
        • How far back does human history stretch in terms of documented melodic creations?
     4. Question_ID=8463  →  有重复query
        • Do the faculty members at Shantou University lead any g

In [1]:
# extract_problematic_to_CSV_with_source.py
import json
from collections import defaultdict

print("正在扫描 train_final.jsonl 和 test_final.jsonl，提取有问题的 Question_ID...")

problematic = []

def load_jsonl_with_source(path, source_name):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                obj = json.loads(line)
                obj["_source_file"] = source_name   # 临时标记来源
                records.append(obj)
    return records

# 加载并标记来源
train_data = load_jsonl_with_source("train_final.jsonl", "train")
test_data  = load_jsonl_with_source("test_final.jsonl",  "test")

all_data = train_data + test_data

# 按 Question_ID 分组
groups = defaultdict(list)
for entry in all_data:
    qid = str(entry.get("Question_ID", ""))
    groups[qid].append(entry)

print(f"总共 {len(groups)} 个唯一 Question_ID，正在检查...")

for qid, entries in groups.items():
    queries = [str(e.get("user_query", "")).strip() for e in entries]
    answers = [str(e.get("true_a_i", "")).strip() for e in entries]

    n_total = len(entries)
    n_unique_q = len(set(queries))
    n_unique_a = len(set(answers))

    # 有问题的情况
    if n_total == 1 or n_unique_q < n_total or n_unique_a > 1:
        # 优先选有 true_q_i 的那一行
        chosen = None
        for e in entries:
            if str(e.get("true_q_i", "")).strip() not in ("", "nan", "None", None):
                chosen = e
                break
        if not chosen:
            chosen = entries[0]

        fixed = chosen.copy()
        # 替换 user_query 为 true_q_i
        fixed["user_query"] = str(fixed.get("true_q_i", "")).strip()
        # 正式添加 source 列（train 或 test）
        fixed["source"] = fixed.pop("_source_file")  # 转正

        problematic.append(fixed)

# 转成 DataFrame 并保存为 CSV
import pandas as pd
df_out = pd.DataFrame(problematic)

# 确保 source 列在最前面（方便你看）
cols = ["source", "Question_ID"] + [c for c in df_out.columns if c not in ["source", "Question_ID"]]
df_out = df_out[cols]

output_csv = "problematic_questions_with_source.csv"
df_out.to_csv(output_csv, index=False, encoding="utf-8")

print("\n完成！")
print(f"共提取 {len(df_out)} 个有问题的 Question_ID")
print(f"已保存为 CSV → {output_csv}")
print("列包含：source（train/test） + 所有原始字段")
print("user_query 已全部替换为 true_q_i，准备好喂 LLM 重新生成多样化 query！")

正在扫描 train_final.jsonl 和 test_final.jsonl，提取有问题的 Question_ID...
总共 8807 个唯一 Question_ID，正在检查...

完成！
共提取 1918 个有问题的 Question_ID
已保存为 CSV → problematic_questions_with_source.csv
列包含：source（train/test） + 所有原始字段
user_query 已全部替换为 true_q_i，准备好喂 LLM 重新生成多样化 query！
